# ocr.ipynb

This file loads each historical document from `data/` and parses it using OCR to extract text for subsequent feature extraction and analysis.

In [3]:
# Compare OCR frameworks on one page: Tesseract vs PaddleOCR
import os
os.environ.update({
    "TQDM_DISABLE": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "GLOG_minloglevel": "3",
    "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK": "True",
})
import io, time, logging, warnings, contextlib
logging.disable(logging.CRITICAL)
warnings.filterwarnings("ignore")

from pathlib import Path
import fitz
import numpy as np
from PIL import Image

PAGE = 0  # 0-indexed

pdf = sorted(Path("data").glob("*.pdf"))[0]
print("PDF:", pdf)
pix = fitz.open(pdf)[PAGE].get_pixmap(dpi=300)
img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")

def tess():
    import pytesseract
    return pytesseract.image_to_string(img, config="--psm 6")

def paddle():
    from paddleocr import PaddleOCR
    ocr = PaddleOCR(enable_mkldnn=False,
                    use_doc_orientation_classify=False,
                    use_doc_unwarping=False,
                    use_textline_orientation=False)
    return "\n".join(ocr.predict(np.array(img))[0]["rec_texts"])

results, times = {}, {}
for name, fn in [("tesseract", tess), ("paddleocr", paddle)]:
    t = time.time()
    with open(os.devnull, "w") as dn, contextlib.redirect_stdout(dn), contextlib.redirect_stderr(dn):
        try:
            results[name] = fn()
        except Exception as e:
            results[name] = f"[FAILED] {type(e).__name__}: {e}"
    times[name] = time.time() - t

for name, text in results.items():
    print(f"\n===== {name} ({times[name]:.1f}s) =====\n{text}")

PDF: data\SERIALSET-00507_00_00-009-0039-0000.pdf

===== tesseract (3.8s) =====
30th Coneress, _[SENATE.] Exxcurive,
1st Session. ‘ No. 39:5
x REPORT
OR. THE yo ae
COMMISSIONER OF THE GENERAL LAND OFFICE, | :
, COMMUNICATING : | \
We An abstract of permits granted under the acts for the armed occu-
; eo pation of Florida.
Apri 28, 1848. :
Read, and ordered to be printed, and that 1,500 additional copies be-printed for the General
Land Office. ;
{ Geyurat Lanp Orricn, April 27, 1848.
Sra: In accordance with the directions in the resolution of the
Senate, of the 3d instant, [ have the honor to transmit, herewith, an
abstract or list, marked No. 1, *‘ of all permits granted under the
, acts for the armed occupation of Florida, specifying, in different
columns, the number and date of each permit, names of ‘settlers,
designation of land, number of notice, date, person to whom permit
was delivered, whether head of a: family or single man; whether
land designated was surveyed, and when, and i